# 01 · MedGemma 1.5 4B: structured chest X-ray anatomical extraction

> **Research use only.** Generated radiographs and CT volumes are synthetic and patient-conditioned. They are
> estimates, not acquisitions of the patient and not clinical ground truth. Never use them for diagnosis or treatment.
> Upload only images you are authorised to use, and remove burned-in identifiers first.

**What this notebook does:** runs `google/medgemma-1.5-4b-it` on public CC0 radiographs. It produces the
`radiovue.anatomy.v1` JSON schema (view, structures, landmarks, relationships, findings, uncertainties), the official
MedGemma 1.5 anatomy bounding boxes (frontal only), and a cross-check of those boxes against a dedicated
segmentation model.

**Requirements:** a GPU (L4 24 GB recommended; a T4 should work in bf16 but slower; *unverified*), plus a Hugging Face
account that has accepted the HAI-DEF terms on the model page, with `HF_TOKEN` in Colab Secrets.

**Design rules:** semantic text and spatial output are separate. Coordinates in the semantic answer are rejected.
Boxes are approximate and are not masks. Decoding is greedy, and prompts are hashed so runs are reproducible.

In [ ]:
#@title 1. Configuration { display-mode: "form" }
REPO_URL = "https://github.com/sanafathima-00/RadioVue.git"  #@param {type:"string"}
REPO_BRANCH = "claude/single-xray-vlm-pipeline"  #@param {type:"string"}
#@markdown If the pipeline code is not pushed yet, upload `radiovue_colab_bundle.zip` (see SETUP_GUIDE.md) to this Drive path:
CODE_BUNDLE_ON_DRIVE = "RadioVue/code/radiovue_colab_bundle.zip"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/RadioVue"  #@param {type:"string"}
DVG_CHECKPOINTS_ON_DRIVE = DRIVE_ROOT + "/Volumea/models/DVG-Diffusion"
PARALLAX_MODEL_ON_DRIVE = DRIVE_ROOT + "/Parallax/models/DiT-fb-512"  # used if present, else downloaded from Hugging Face
OUTPUT_ROOT = DRIVE_ROOT + "/outputs/single_xray_pipeline"
REPO_DIR = "/content/RadioVue"

In [ ]:
#@title 2. GPU, Drive and code
import os, sys, subprocess, zipfile, importlib.util
from pathlib import Path

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(smi.stdout[:700] or smi.stderr)
assert smi.returncode == 0, "No GPU. Runtime > Change runtime type > GPU (L4/A100 recommended, T4 minimum)."

from google.colab import drive
drive.mount("/content/drive")

bundle = Path("/content/drive/MyDrive") / CODE_BUNDLE_ON_DRIVE
if not Path(REPO_DIR, "radiovue_pipeline").exists():
    if bundle.is_file():
        Path(REPO_DIR).mkdir(parents=True, exist_ok=True)
        zipfile.ZipFile(bundle).extractall(REPO_DIR)
        print("Code from Drive bundle:", bundle)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
assert Path(REPO_DIR, "radiovue_pipeline").exists(), (
    "radiovue_pipeline/ not found. Push the branch that contains it, or upload the code bundle (SETUP_GUIDE.md).")
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
rev = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Code revision:", rev or "(bundle, see BUNDLE_MANIFEST.json)")

# colab_setup has no third-party imports, so it can run before packages are installed.
spec = importlib.util.spec_from_file_location("colab_setup", f"{REPO_DIR}/radiovue_pipeline/colab_setup.py")
cs = importlib.util.module_from_spec(spec); spec.loader.exec_module(cs)
GPU = cs.gpu_report()

In [ ]:
#@title 3. Install dependencies and stage models (first run ~10-15 min)
cs.install_runtime(parallax=False, dvg=False, tools=True)
cs.pip("transformers>=4.50", "accelerate>=1.1", extra=["-U"])  # this notebook does not load Parallax
HF_TOKEN = cs.hf_token_from_colab_secrets()  # value is never printed
assert HF_TOKEN, "Add HF_TOKEN in Colab Secrets (key icon) and enable notebook access."
cs.check_gated_access("google/medgemma-1.5-4b-it")

In [ ]:
#@title Public CC0 test radiographs (same subject, same session; Wikimedia Commons, user Stillwaterising, 2010)
import hashlib, urllib.request
PUBLIC = {
    "pa_cc0_2010.png": ("https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png",
                        "a43431b089a6dfab031c10eceef6c821e4f326913bf9015f73f4a1f0446d2be1"),
    "lateral_cc0_2010.png": ("https://upload.wikimedia.org/wikipedia/commons/9/9d/Chest_Xray_Lateral_3-8-2010.png",
                             "d07a306fb285a8456739e0349fb208b64ade12eb4bb84bc09889c7b3ec7f99a7"),
}
PUBLIC_DIR = Path("/content/public_cc0"); PUBLIC_DIR.mkdir(exist_ok=True)
for name, (url, digest) in PUBLIC.items():
    path = PUBLIC_DIR / name
    if not path.exists():
        req = urllib.request.Request(url, headers={"User-Agent": "RadioVueResearch/0.1"})
        path.write_bytes(urllib.request.urlopen(req).read())
    assert hashlib.sha256(path.read_bytes()).hexdigest() == digest, f"Checksum mismatch: {name}"
print("Public test images verified:", sorted(p.name for p in PUBLIC_DIR.iterdir()))

In [ ]:
#@title 4. Load MedGemma (bf16) and the segmentation model
import json, time, torch
from radiovue_pipeline.anatomy_extraction import MedGemmaClient, run_extraction, compare_boxes_with_segmentation
from radiovue_pipeline.segmentation import AnatomySegmenter
from radiovue_pipeline.io_utils import load_xray
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
client = MedGemmaClient("google/medgemma-1.5-4b-it", max_new_tokens=1500)
print(f"Loaded in {time.time()-t0:.0f}s; revision={client.revision}; VRAM after load: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")
segmenter = AnatomySegmenter()
OUT = Path(OUTPUT_ROOT) / "medgemma_examples"; OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
#@title 5. Run structured extraction on the public PA and lateral images
results, timings = {}, {}
for name, frontal in [("pa_cc0_2010.png", True), ("lateral_cc0_2010.png", False)]:
    image = load_xray(PUBLIC_DIR / name).image
    t0 = time.time()
    extraction = run_extraction(image, client.generate, is_frontal=frontal, model_revision=client.revision)
    timings[name] = round(time.time() - t0, 1)
    if frontal:
        seg = segmenter.run(image, True, OUT / f"{name}_segmentation")
        extraction["segmentation"] = seg
        extraction["box_vs_mask_iou"] = compare_boxes_with_segmentation(extraction, seg["measurements_px"])
    results[name] = extraction
    (OUT / f"{Path(name).stem}_extraction.json").write_text(json.dumps(extraction, indent=2, default=str))
print("Seconds per image:", timings)
print("Peak VRAM GiB:", round(torch.cuda.max_memory_allocated()/1024**3, 2))
for name, r in results.items():
    print("\n==", name, "| view:", r["image_metadata"], "| rejected spatial claims:", len(r["rejected_spatial_claims"]))
    print(json.dumps({k: r[k] for k in ("anatomical_structures", "spatial_relationships", "uncertainties")}, indent=1)[:2500])

In [ ]:
#@title 6. Box quality vs dedicated segmentation (cross-model agreement, not ground truth)
import pandas as pd
display(pd.DataFrame(results["pa_cc0_2010.png"]["box_vs_mask_iou"]))

In [ ]:
#@title 7. Draw MedGemma boxes (approximate) on the PA image
from PIL import ImageDraw
img = load_xray(PUBLIC_DIR / "pa_cc0_2010.png").image.convert("RGB")
draw = ImageDraw.Draw(img)
for item in results["pa_cc0_2010.png"]["localization"]["items"]:
    for b in item["boxes"]:
        if b["box_xyxy_px_original"]:
            draw.rectangle(b["box_xyxy_px_original"], outline=(255, 0, 0), width=6)
            draw.text((b["box_xyxy_px_original"][0] + 8, b["box_xyxy_px_original"][1] + 8), item["query"], fill=(255, 255, 0))
img.save(OUT / "pa_medgemma_boxes.png"); img.resize((img.width // 3, img.height // 3))

In [ ]:
#@title 8. Reproducibility check: identical prompt twice -> identical output under greedy decoding
from radiovue_pipeline.anatomy_extraction import VIEW_PROMPT
pa = load_xray(PUBLIC_DIR / "pa_cc0_2010.png").image.convert("RGB")
a, b = client.generate(pa, VIEW_PROMPT).response, client.generate(pa, VIEW_PROMPT).response
print("Identical:", a == b)
(OUT / "environment.json").write_text(json.dumps({"gpu": GPU, "torch": torch.__version__,
    "transformers": __import__("transformers").__version__, "model_revision": client.revision,
    "seconds_per_image": timings, "peak_vram_gib": round(torch.cuda.max_memory_allocated()/1024**3, 2),
    "greedy_reproducible": a == b}, indent=2))
print("Saved to", OUT)